In [1]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT 
from psycopg2.extras import execute_batch

import pandas as pd

In [17]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: '))
with open('pwd.txt', 'r') as file:
    pwd = file.read()
conn = psycopg2.connect(
        dbname='postgres',
        user='postgres',
        password=pwd,
        host='localhost',
        port='5432')

In [18]:
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

In [19]:
cur = conn.cursor()

In [20]:
# ИЛОНА-------------------------------------------------
df = pd.read_csv('flight_data_.csv')

mapping = {'int64' : 'INTEGER',
        'str' : 'VARCHAR(100)',
        'object': 'VARCHAR(100)',
        'float64' : 'NUMERIC(10, 2)',
        'bool' : 'BOOLEAN'}
postgre_dtypes = df.dtypes.reset_index(drop=False).iloc[:, 1].astype(str)
postgre_dtypes = postgre_dtypes.apply(lambda x: mapping[x])

cols = ', '.join([f'"{col}"' for col in df.columns])
columns = ''
for c, t in zip(df.columns, postgre_dtypes):
    columns += f'"{c}" {t}, '
columns = columns.strip(', ')
vals = ', '.join(['%s'] * len(df.columns))

cur.execute(f'''DROP TABLE IF EXISTS data; 
            CREATE TABLE data ({columns});''')
cur.executemany(f'INSERT INTO data ({cols}) VALUES ({vals});', [row for row in df.itertuples(index=False)])

In [21]:
#ГЛЕБ-------------------------------------------------
# Convert the flight date to date type
# cur.execute('''ALTER TABLE data 
# ALTER COLUMN "Departure Date" TYPE TIMESTAMP 
# USING TO_TIMESTAMP("Departure Date", 'YYYY-MM-DD HH24:MI:SS');''')

#Create a “cleaned table” in case there is no value in the delay column (to use COALESCE)
cur.execute("""
DROP TABLE IF EXISTS data_clean;

CREATE TABLE data_clean AS
SELECT 
    *,
    COALESCE("Delay Minutes", 0) AS "Delay Minutes Clean"
FROM data;
""")

#Create a flight schedule table
cur.execute('''
DROP TABLE IF EXISTS flights_schedule;
CREATE TABLE flights_schedule AS
SELECT
    ROW_NUMBER() OVER (ORDER BY "Route", "Departure Date") AS "Flight ID",
    "Route",
    "Departure Date",
    MAX("Delay Minutes Clean") AS "Delay Minutes Clean",
    COUNT(*) AS "Passenger Count",
    AVG("Ticket Price") AS "Avg Ticket Price"
FROM data_clean
GROUP BY "Route", "Departure Date";
''')

#Create a passenger table
cur.execute('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT DISTINCT ON ("Customer ID")
    "Customer ID",
    "Name",
    COUNT(*) OVER(PARTITION BY "Customer ID") AS "Flights count",
    "Frequent Flyer Status",
    "Loyalty Points"
FROM data_clean
ORDER BY "Customer ID", "Departure Date" DESC;
''')

In [22]:
#ИЛОНА-------------------------------------------------
# First Photo*
# Create a table of tickets (so we can use different joins later)
# These joins won't give us any useful results, since everything is derived from a single shared dataset in which all rows are filled
cur.execute('''
DROP TABLE IF EXISTS tickets;
CREATE TABLE tickets AS
SELECT 
    f."Flight ID",           
    d."Customer ID",         
    d."Booking Class",       
    d."Ticket Price",       
    d."Competitor Price",    
    d."Demand",              
    d."Profitability"     
FROM data_clean d
JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
# Here we've basically created tickets so that every passenger has a flight number (if a passenger doesn't have a flight or if the flight is empty, it's skipped)
# These are the kinds of tickets we have. here we only include those who have tickets
df_tickets = pd.read_sql("""
SELECT *
FROM tickets
LIMIT 10;
""", conn)

df_tickets.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\2879178922.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tickets = pd.read_sql("""


,Flight ID,Customer ID,Booking Class,Ticket Price,Competitor Price,Demand,Profitability
0,41,3769,Business,370.64,382.95,-0.93,0.63
1,1,3529,Economy,114.53,394.58,-1.01,1.27
2,47,1303,Economy,164.47,479.83,1.76,1.14
3,32,2965,Economy,318.90,286.30,-0.52,1.13
4,14,8779,Economy,389.97,407.46,-0.67,1.22


In [23]:
#ИЛОНА-------------------------------------------------
#Second Photo*
# If we use the left join, not all passengers would have a flight number if, for example, some flights were missing from the flight schedule
# This way, we have all passenger records + the numbers of existing flights
cur.execute('''
DROP TABLE IF EXISTS tickets_left;
CREATE TABLE tickets_left AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    d."Ticket Price"
FROM data_clean d
LEFT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_ticketsleft = pd.read_sql("""
SELECT *
FROM tickets_left
LIMIT 10;
""", conn)

df_tickets.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\1021427955.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ticketsleft = pd.read_sql("""


,Flight ID,Customer ID,Booking Class,Ticket Price,Competitor Price,Demand,Profitability
0,41,3769,Business,370.64,382.95,-0.93,0.63
1,1,3529,Economy,114.53,394.58,-1.01,1.27
2,47,1303,Economy,164.47,479.83,1.76,1.14
3,32,2965,Economy,318.90,286.30,-0.52,1.13
4,14,8779,Economy,389.97,407.46,-0.67,1.22


In [24]:
#ИЛОНА-------------------------------------------------
#Second Photo*
#Here the right join leaves all the passengers with scheduled flights 
cur.execute('''
DROP TABLE IF EXISTS flights_with_passengers;
CREATE TABLE flights_with_passengers AS
SELECT 
    f."Flight ID",
    f."Route",
    f."Departure Date",
    f."Passenger Count",
    d."Customer ID"
FROM data_clean d
RIGHT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_flights_with_passengers = pd.read_sql("""
SELECT *
FROM flights_with_passengers
LIMIT 10;
""", conn)

df_flights_with_passengers.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\3358426279.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_flights_with_passengers = pd.read_sql("""


,Flight ID,Route,Departure Date,Passenger Count,Customer ID
0,41,MEL-BNE,2023-05-02 20:11:09,12,3769
1,1,BNE-SYD,2023-04-21 00:10:14,14,3529
2,47,MEL-BNE,2023-05-12 15:16:31,11,1303
3,32,BNE-SYD,2023-06-13 20:53:09,11,2965
4,14,BNE-SYD,2023-05-15 23:06:14,17,8779


In [25]:
#ИЛОНА-------------------------------------------------
# Here we take all records with empty flights + passengers with no flights + exists both the passenger and the flight (filled out ticket). 
cur.execute('''
DROP TABLE IF EXISTS full_join;
CREATE TABLE full_join AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    COALESCE(f."Route", d."Route") AS "Route"
FROM data_clean d
FULL OUTER JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_full = pd.read_sql("""
SELECT *
FROM full_join
LIMIT 10;
""", conn)

df_full.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\2315904870.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_full = pd.read_sql("""


,Flight ID,Customer ID,Booking Class,Route
0,41,3769,Business,MEL-BNE
1,1,3529,Economy,BNE-SYD
2,47,1303,Economy,MEL-BNE
3,32,2965,Economy,BNE-SYD
4,14,8779,Economy,BNE-SYD


In [26]:
#ГЛАША -------------------------------------------------
# Table showing 3-day rolling averages and standard deviation
cur.execute('''
DROP TABLE IF EXISTS rolling_3days;
CREATE TABLE rolling_3days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 3 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 3 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 3 THEN "Rolling Avg 3 Days" ELSE NULL END AS "Rolling Avg 3 Days",
    CASE WHEN window_size = 3 THEN "Rolling StdDev 3 Days" ELSE NULL END AS "Rolling StdDev 3 Days",
    CASE 
        WHEN window_size = 3 AND "Rolling Avg 3 Days" > LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 3 AND "Rolling Avg 3 Days" < LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 3 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 3;''')
df_rolling = pd.read_sql("""
SELECT *
FROM rolling_3days
LIMIT 10;
""", conn)

df_rolling.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\3678608691.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rolling = pd.read_sql("""


,Flight ID,Route,Departure Date,Passenger Count,Rolling Avg 3 Days,Rolling StdDev 3 Days,Demand Trend
0,3,BNE-SYD,2023-04-25 21:17:18,12,12.67,1.15,STABLE
1,4,BNE-SYD,2023-04-30 11:39:08,10,11.33,1.15,DECLINING
2,5,BNE-SYD,2023-05-01 01:09:21,14,12.00,2.00,INCREASING
3,6,BNE-SYD,2023-05-05 15:27:08,17,13.67,3.51,INCREASING
4,7,BNE-SYD,2023-05-06 12:36:14,14,15.00,1.73,INCREASING


In [27]:
#ГЛАША -------------------------------------------------
# Table showing 6-day rolling averages and standard deviation
cur.execute('''
DROP TABLE IF EXISTS rolling_6days;
CREATE TABLE rolling_6days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 6 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 6 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 6 THEN "Rolling Avg 6 Days" ELSE NULL END AS "Rolling Avg 6 Days",
    CASE WHEN window_size = 6 THEN "Rolling StdDev 6 Days" ELSE NULL END AS "Rolling StdDev 6 Days",
    CASE 
        WHEN window_size = 6 AND "Rolling Avg 6 Days" > LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 6 AND "Rolling Avg 6 Days" < LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 6 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 6;''')
df_rolling_6 = pd.read_sql("""
SELECT *
FROM rolling_6days
LIMIT 10;
""", conn)

df_rolling_6.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\139898736.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rolling_6 = pd.read_sql("""


,Flight ID,Route,Departure Date,Passenger Count,Rolling Avg 6 Days,Rolling StdDev 6 Days,Demand Trend
0,6,BNE-SYD,2023-05-05 15:27:08,17,13.17,2.40,STABLE
1,7,BNE-SYD,2023-05-06 12:36:14,14,13.17,2.40,STABLE
2,8,BNE-SYD,2023-05-07 23:32:44,16,13.83,2.56,INCREASING
3,9,BNE-SYD,2023-05-10 10:06:19,15,14.33,2.42,INCREASING
4,10,BNE-SYD,2023-05-11 22:47:43,15,15.17,1.17,INCREASING


In [28]:
#ГЛАША -------------------------------------------------
# Table showing 9-day rolling averages and standard deviation
cur.execute('''
DROP TABLE IF EXISTS rolling_9days;
CREATE TABLE rolling_9days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 9 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 9 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 9 THEN "Rolling Avg 9 Days" ELSE NULL END AS "Rolling Avg 9 Days",
    CASE WHEN window_size = 9 THEN "Rolling StdDev 9 Days" ELSE NULL END AS "Rolling StdDev 9 Days",
    CASE 
        WHEN window_size = 9 AND "Rolling Avg 9 Days" > LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 9 AND "Rolling Avg 9 Days" < LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 9 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 9;''')
df_rolling_9 = pd.read_sql("""
SELECT *
FROM rolling_9days
LIMIT 10;
""", conn)

df_rolling_9.head()

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\1269873680.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rolling_9 = pd.read_sql("""


,Flight ID,Route,Departure Date,Passenger Count,Rolling Avg 9 Days,Rolling StdDev 9 Days,Demand Trend
0,9,BNE-SYD,2023-05-10 10:06:19,15,13.78,2.17,STABLE
1,10,BNE-SYD,2023-05-11 22:47:43,15,13.89,2.20,INCREASING
2,11,BNE-SYD,2023-05-13 10:40:25,15,14.22,2.11,INCREASING
3,12,BNE-SYD,2023-05-15 04:40:39,14,14.44,1.94,INCREASING
4,13,BNE-SYD,2023-05-15 05:57:12,15,15.00,1.00,INCREASING


In [ ]:
#КАТЯ ---------------------------------------
cur.execute("""
ALTER TABLE flights_schedule
ALTER COLUMN "Departure Date" TYPE TIMESTAMP
USING "Departure Date"::timestamp;
""")
cur.execute('ALTER TABLE flights_schedule ADD COLUMN "Delay Category" VARCHAR(30);')
cur.execute('ALTER TABLE flights_schedule ADD COLUMN "On Time Flag" INTEGER;')
cur.execute('''UPDATE flights_schedule
SET 
    "Delay Category" = CASE
        WHEN "Delay Minutes Clean" < 15  THEN 'on-time'
        WHEN "Delay Minutes Clean" BETWEEN 15 AND 60 THEN 'minor delay'
        WHEN "Delay Minutes Clean" BETWEEN 60 AND 180 THEN 'major delay'
        WHEN "Delay Minutes Clean" > 180 THEN 'severe delay'
        ELSE 'cancellation' 
    END,
    "On Time Flag" = CASE WHEN "Delay Minutes Clean" < 15 THEN 1 ELSE 0 END;''')

cur.execute('''CREATE TABLE rolling_on_time_percentage AS
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Delay Category",
    "On Time Flag",
    ROUND(
        AVG("On Time Flag") OVER (
            PARTITION BY "Route"
            ORDER BY "Departure Date"
            RANGE BETWEEN INTERVAL '7 days' PRECEDING AND CURRENT ROW
        ) * 100, 2
    ) AS "Rolling On-Time % (last 7 days)"
FROM flights_schedule
ORDER BY "Route", "Departure Date";''')

In [30]:
df_on_time = pd.read_sql("""
SELECT *
FROM rolling_on_time_percentage
LIMIT 10;
""", conn)

df_on_time.head(9)

C:\Users\Daria\AppData\Local\Temp\ipykernel_10216\337746933.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_on_time = pd.read_sql("""


,Flight ID,Route,Departure Date,Delay Category,On Time Flag,Rolling On-Time % (last 7 days)
0,1,BNE-SYD,2023-04-21 00:10:14,minor delay,0,0.00
1,2,BNE-SYD,2023-04-23 14:45:53,minor delay,0,0.00
2,3,BNE-SYD,2023-04-25 21:17:18,major delay,0,0.00
3,4,BNE-SYD,2023-04-30 11:39:08,minor delay,0,0.00
4,5,BNE-SYD,2023-05-01 01:09:21,major delay,0,0.00
5,6,BNE-SYD,2023-05-05 15:27:08,on-time,1,33.33
6,7,BNE-SYD,2023-05-06 12:36:14,minor delay,0,25.00
7,8,BNE-SYD,2023-05-07 23:32:44,major delay,0,25.00
8,9,BNE-SYD,2023-05-10 10:06:19,minor delay,0,25.00


In [31]:
cur.close()
conn.close()